In [1]:
import pandas as pd
df = pd.read_parquet('C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_exer3_processed.parquet', engine='pyarrow')

In [2]:
df.shape

(124494, 96)

In [ ]:
# Split temporal y por device con estratificación
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# Asegurar date_trunc
if 'date_trunc' not in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df['date_trunc'] = df['date'].astype(str).str[:7]

# Filtro temporal
train_months = ["2015-01","2015-02","2015-03","2015-04","2015-05","2015-06","2015-07"]
oot_months = ["2015-08","2015-09","2015-10","2015-11"]

period_train = df[df['date_trunc'].isin(train_months)].copy()
period_oot = df[df['date_trunc'].isin(oot_months)].copy()

print(f"Registros periodo train (2015-01..2015-07): {len(period_train)}")
print(f"Registros periodo OOT (2015-08..2015-11): {len(period_oot)}")

# 2) Seleccionar 25% de devices de OOT estratificado por failure
oot_devices = period_oot['device'].unique()
rng = np.random.default_rng(42)

# Estratificar por failure rate por device
device_failure_rates = period_oot.groupby('device')['failure'].mean().reset_index()
device_failure_rates['strata'] = pd.qcut(device_failure_rates['failure'], q=5, duplicates='drop', labels=False)

# Seleccionar 25% de devices manteniendo proporción de failure
control_n_devices = max(1, int(0.25 * len(oot_devices)))
control_devices = []

for stratum in device_failure_rates['strata'].unique():
    stratum_devices = device_failure_rates[device_failure_rates['strata'] == stratum]['device'].tolist()
    stratum_control_n = max(1, int(0.25 * len(stratum_devices)))
    stratum_control = rng.choice(stratum_devices, size=min(stratum_control_n, len(stratum_devices)), replace=False)
    control_devices.extend(stratum_control)

control_devices = set(control_devices)

# Excluir devices de control del training para evitar dataleakage
period_train_clean = period_train[~period_train['device'].isin(control_devices)].copy()

print(f"Devices OOT totales: {len(oot_devices)}")
print(f"Devices control seleccionados: {len(control_devices)}")
print(f"Registros training original: {len(period_train)}")
print(f"Registros training después de excluir control: {len(period_train_clean)}")

# Crear splits de OOT
control_df = period_oot[period_oot['device'].isin(control_devices)].copy()
oos_df = period_oot[~period_oot['device'].isin(control_devices)].copy()

print(f"Registros control: {len(control_df)} | Registros OOS: {len(oos_df)}")

# 3) Split del periodo_train_clean en 70/30 con control de leakage por device y estratificación por failure

def stratified_group_train_test_split(df_in, label_col, group_col, test_size=0.30, random_state=42):
    X = df_in.index.values
    y = df_in[label_col].values
    groups = df_in[group_col].values

    # Etiqueta estratificada por grupo: proporción de clase 1 por device, binned
    grp = (
        df_in.groupby(group_col)[label_col]
        .mean()
        .rename('failure_rate')
        .reset_index()
    )
    # Mapear a bins para estratificar grupos
    grp['strata'] = pd.qcut(grp['failure_rate'], q=min(5, grp['failure_rate'].nunique()), duplicates='drop', labels=False)
    strata = df_in[group_col].map(grp.set_index(group_col)['strata']).values

    best = None
    best_diff = 1.0
    n_splits = 10
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for train_idx, test_idx in cv.split(X, y=strata, groups=groups):
        ratio = len(test_idx) / len(X)
        diff = abs(ratio - test_size)
        if diff < best_diff:
            best = (train_idx, test_idx)
            best_diff = diff
    if best is None:
        raise RuntimeError("No se pudo obtener un split estratificado por grupos.")
    train_idx, test_idx = best
    return df_in.iloc[train_idx].copy(), df_in.iloc[test_idx].copy()

train_df, test_df = stratified_group_train_test_split(period_train_clean, label_col='failure', group_col='device', test_size=0.30, random_state=42)

# 4) Verificar distribución por failure en cada muestra

def dist(y):
    v = pd.Series(y).value_counts(normalize=True).sort_index()
    return {int(k): float(v.get(k, 0.0)) for k in [0,1]}

print("\nDistribución por failure (proporción):")
print({
    'train': dist(train_df['failure']),
    'test': dist(test_df['failure']),
    'control': dist(control_df['failure']),
    'oos': dist(oos_df['failure'])
})

# Reporte final
print("\nResumen de tamaños:")
print({
    'train_rows': len(train_df),
    'test_rows': len(test_df),
    'control_rows': len(control_df),
    'oos_rows': len(oos_df)
})

# Variables de salida clave
SPLITS = {
    'train': train_df,
    'test': test_df,
    'control': control_df,
    'oos': oos_df
}
print("\nSPLITS = {'train','test','control','oos'} listos.")


In [ ]:
# Separar devices por periodos temporalmente
train_devices = set(df[df['date_trunc'].isin(["2015-01","2015-02","2015-03","2015-04","2015-05","2015-06","2015-07"])]['device'].unique())
oot_devices = set(df[df['date_trunc'].isin(["2015-08","2015-09","2015-10","2015-11"])]['device'].unique())
print(f"Devices en training: {len(train_devices)} | Devices en OOT: {len(oot_devices)} | Intersección: {len(train_devices & oot_devices)}")


In [ ]:
# Validación simple de solapamiento
train_in_oot = len(train_devices & oot_devices)
porcentaje = (train_in_oot / len(train_devices)) * 100
print(f"Devices de training que están en OOT: {train_in_oot} de {len(train_devices)} ({porcentaje:.1f}%)")


In [4]:
df.shape

(124494, 96)

In [5]:
df.head()

,date,device,failure,attribute1,attribute2,attribute3,attribute4,attribute5,attribute6,attribute7,...,attribute3_change_rate,attribute3_cumulative_change,attribute4_change_rate,attribute4_cumulative_change,attribute7_change_rate,attribute7_cumulative_change,attribute8_change_rate,attribute8_cumulative_change,attribute9_change_rate,attribute9_cumulative_change
0,2015-01-01,S1F01085,0,215630672,56,0,52,6,407438,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1163,2015-01-02,S1F01085,0,1650864,56,0,52,6,407438,0,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0
2326,2015-01-03,S1F01085,0,124017368,56,0,52,6,407438,0,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0
3489,2015-01-04,S1F01085,0,128073224,56,0,52,6,407439,0,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0
4651,2015-01-05,S1F01085,0,97393448,56,0,52,6,408114,0,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0


In [6]:
df['date_trunc'] = df['date'].astype(str).str[:7]

In [9]:
pd.crosstab(df['date_trunc'], df['failure'], normalize='index').sort_values(by='date_trunc', ascending=True)

failure,0,1
date_trunc,,
2015-01,0.999041,0.000959
2015-02,0.999282,0.000718
2015-03,0.999546,0.000454
2015-04,0.999251,0.000749
2015-05,0.998147,0.001853
2015-06,0.999427,0.000573
2015-07,0.998481,0.001519
2015-08,0.999521,0.000479
2015-09,1.000000,0.000000


In [10]:
df.shape

(124494, 97)

In [14]:

# Separar devices por periodos temporalmente
train_devices = set(df[df['date_trunc'].isin(["2015-01","2015-02","2015-03","2015-04","2015-05","2015-06","2015-07"])]['device'].unique())
oot_devices = set(df[df['date_trunc'].isin(["2015-08","2015-09","2015-10","2015-11"])]['device'].unique())
print(f"Devices en training: {len(train_devices)} | Devices en OOT: {len(oot_devices)} | Intersección: {len(train_devices & oot_devices)}")

Devices en training: 1169 | Devices en OOT: 334 | Intersección: 334


In [22]:
# Comprobar solapamiento entre muestras usando los devices ya calculados
print("=== ANÁLISIS DE SOLAPAMIENTO ENTRE MUESTRAS ===")

# Usar los devices ya calculados en la celda anterior
print(f"Devices en training: {len(train_devices)}")
print(f"Devices en OOT: {len(oot_devices)}")
print(f"Intersección Train-OOT: {len(train_devices & oot_devices)}")

print(f"\n=== ANÁLISIS DE SOLAPAMIENTO ===")
print(f"Devices que aparecen en ambos periodos: {len(train_devices & oot_devices)}")
print(f"Devices solo en training: {len(train_devices - oot_devices)}")
print(f"Devices solo en OOT: {len(oot_devices - train_devices)}")

print(f"\n=== IMPLICACIONES PARA EL SPLIT ===")
print(f"Total devices únicos: {len(train_devices | oot_devices)}")
print(f"Porcentaje de solapamiento: {len(train_devices & oot_devices) / len(train_devices | oot_devices) * 100:.1f}%")

if len(train_devices & oot_devices) > 0:
    print("⚠️ Hay devices que aparecen en ambos periodos - esto es normal")
    print("✅ El split debe funcionar correctamente")
else:
    print("ℹ️ No hay devices en ambos periodos - split temporal perfecto")



# Validación simple de solapamiento
train_in_oot = len(train_devices & oot_devices)
porcentaje = (train_in_oot / len(train_devices)) * 100
print(f"Devices de training que están en OOT: {train_in_oot} de {len(train_devices)} ({porcentaje:.1f}%)")

=== ANÁLISIS DE SOLAPAMIENTO ENTRE MUESTRAS ===
Devices en training: 1169
Devices en OOT: 334
Intersección Train-OOT: 334

=== ANÁLISIS DE SOLAPAMIENTO ===
Devices que aparecen en ambos periodos: 334
Devices solo en training: 835
Devices solo en OOT: 0

=== IMPLICACIONES PARA EL SPLIT ===
Total devices únicos: 1169
Porcentaje de solapamiento: 28.6%
⚠️ Hay devices que aparecen en ambos periodos - esto es normal
✅ El split debe funcionar correctamente
Devices de training que están en OOT: 334 de 1169 (28.6%)


In [23]:
# Split temporal y por device con estratificación
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# Asegurar date_trunc
if 'date_trunc' not in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df['date_trunc'] = df['date'].astype(str).str[:7]

# Filtro temporal
train_months = ["2015-01","2015-02","2015-03","2015-04","2015-05","2015-06","2015-07"]
oot_months = ["2015-08","2015-09","2015-10","2015-11"]

period_train = df[df['date_trunc'].isin(train_months)].copy()
period_oot = df[df['date_trunc'].isin(oot_months)].copy()

print(f"Registros periodo train (2015-01..2015-07): {len(period_train)}")
print(f"Registros periodo OOT (2015-08..2015-11): {len(period_oot)}")

# 2) Control de dataleakage: primero seleccionar 25% de devices de OOT, luego excluir esos devices de training
oot_devices = set(period_oot['device'].unique())
rng = np.random.default_rng(42)
control_n_devices = max(1, int(0.25 * len(oot_devices)))
control_devices = set(rng.choice(list(oot_devices), size=min(control_n_devices, len(oot_devices)), replace=False))

# Excluir devices de control del training para evitar dataleakage
period_train_clean = period_train[~period_train['device'].isin(control_devices)].copy()

print(f"Devices OOT totales: {len(oot_devices)}")
print(f"Devices control seleccionados: {len(control_devices)}")
print(f"Registros training original: {len(period_train)}")
print(f"Registros training después de excluir control: {len(period_train_clean)}")

# Crear splits de OOT
control_df = period_oot[period_oot['device'].isin(control_devices)].copy()
oos_df = period_oot[~period_oot['device'].isin(control_devices)].copy()

print(f"Registros control: {len(control_df)} | Registros OOS: {len(oos_df)}")

# 3) Split del periodo_train_clean en 70/30 con control de leakage por device y estratificación por failure

def stratified_group_train_test_split(df_in, label_col, group_col, test_size=0.30, random_state=42):
    X = df_in.index.values
    y = df_in[label_col].values
    groups = df_in[group_col].values

    # Etiqueta estratificada por grupo: proporción de clase 1 por device, binned
    grp = (
        df_in.groupby(group_col)[label_col]
        .mean()
        .rename('failure_rate')
        .reset_index()
    )
    # Mapear a bins para estratificar grupos
    grp['strata'] = pd.qcut(grp['failure_rate'], q=min(5, grp['failure_rate'].nunique()), duplicates='drop', labels=False)
    strata = df_in[group_col].map(grp.set_index(group_col)['strata']).values

    best = None
    best_diff = 1.0
    n_splits = 10
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for train_idx, test_idx in cv.split(X, y=strata, groups=groups):
        ratio = len(test_idx) / len(X)
        diff = abs(ratio - test_size)
        if diff < best_diff:
            best = (train_idx, test_idx)
            best_diff = diff
    if best is None:
        raise RuntimeError("No se pudo obtener un split estratificado por grupos.")
    train_idx, test_idx = best
    return df_in.iloc[train_idx].copy(), df_in.iloc[test_idx].copy()

train_df, test_df = stratified_group_train_test_split(period_train_clean, label_col='failure', group_col='device', test_size=0.30, random_state=42)

# 4) Verificar distribución por failure en cada muestra

def dist(y):
    v = pd.Series(y).value_counts(normalize=True).sort_index()
    return {int(k): float(v.get(k, 0.0)) for k in [0,1]}

print("\nDistribución por failure (proporción):")
print({
    'train': dist(train_df['failure']),
    'test': dist(test_df['failure']),
    'control': dist(control_df['failure']),
    'oos': dist(oos_df['failure'])
})

# Reporte final
print("\nResumen de tamaños:")
print({
    'train_rows': len(train_df),
    'test_rows': len(test_df),
    'control_rows': len(control_df),
    'oos_rows': len(oos_df)
})

# Variables de salida clave
SPLITS = {
    'train': train_df,
    'test': test_df,
    'control': control_df,
    'oos': oos_df
}
print("\nSPLITS = {'train','test','control','oos'} listos.")



# Split temporal y por device con estratificación
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# Asegurar date_trunc
if 'date_trunc' not in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df['date_trunc'] = df['date'].astype(str).str[:7]

# Filtro temporal
train_months = ["2015-01","2015-02","2015-03","2015-04","2015-05","2015-06","2015-07"]
oot_months = ["2015-08","2015-09","2015-10","2015-11"]

period_train = df[df['date_trunc'].isin(train_months)].copy()
period_oot = df[df['date_trunc'].isin(oot_months)].copy()

print(f"Registros periodo train (2015-01..2015-07): {len(period_train)}")
print(f"Registros periodo OOT (2015-08..2015-11): {len(period_oot)}")

# 2) Seleccionar 25% de devices de OOT estratificado por failure
oot_devices = period_oot['device'].unique()
rng = np.random.default_rng(42)

# Estratificar por failure rate por device
device_failure_rates = period_oot.groupby('device')['failure'].mean().reset_index()
device_failure_rates['strata'] = pd.qcut(device_failure_rates['failure'], q=5, duplicates='drop', labels=False)

# Seleccionar 25% de devices manteniendo proporción de failure
control_n_devices = max(1, int(0.25 * len(oot_devices)))
control_devices = []

for stratum in device_failure_rates['strata'].unique():
    stratum_devices = device_failure_rates[device_failure_rates['strata'] == stratum]['device'].tolist()
    stratum_control_n = max(1, int(0.25 * len(stratum_devices)))
    stratum_control = rng.choice(stratum_devices, size=min(stratum_control_n, len(stratum_devices)), replace=False)
    control_devices.extend(stratum_control)

control_devices = set(control_devices)

# Excluir devices de control del training para evitar dataleakage
period_train_clean = period_train[~period_train['device'].isin(control_devices)].copy()

print(f"Devices OOT totales: {len(oot_devices)}")
print(f"Devices control seleccionados: {len(control_devices)}")
print(f"Registros training original: {len(period_train)}")
print(f"Registros training después de excluir control: {len(period_train_clean)}")

# Crear splits de OOT
control_df = period_oot[period_oot['device'].isin(control_devices)].copy()
oos_df = period_oot[~period_oot['device'].isin(control_devices)].copy()

print(f"Registros control: {len(control_df)} | Registros OOS: {len(oos_df)}")

# 3) Split del periodo_train_clean en 70/30 con control de leakage por device y estratificación por failure

def stratified_group_train_test_split(df_in, label_col, group_col, test_size=0.30, random_state=42):
    X = df_in.index.values
    y = df_in[label_col].values
    groups = df_in[group_col].values

    # Etiqueta estratificada por grupo: proporción de clase 1 por device, binned
    grp = (
        df_in.groupby(group_col)[label_col]
        .mean()
        .rename('failure_rate')
        .reset_index()
    )
    # Mapear a bins para estratificar grupos
    grp['strata'] = pd.qcut(grp['failure_rate'], q=min(5, grp['failure_rate'].nunique()), duplicates='drop', labels=False)
    strata = df_in[group_col].map(grp.set_index(group_col)['strata']).values

    best = None
    best_diff = 1.0
    n_splits = 10
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for train_idx, test_idx in cv.split(X, y=strata, groups=groups):
        ratio = len(test_idx) / len(X)
        diff = abs(ratio - test_size)
        if diff < best_diff:
            best = (train_idx, test_idx)
            best_diff = diff
    if best is None:
        raise RuntimeError("No se pudo obtener un split estratificado por grupos.")
    train_idx, test_idx = best
    return df_in.iloc[train_idx].copy(), df_in.iloc[test_idx].copy()

train_df, test_df = stratified_group_train_test_split(period_train_clean, label_col='failure', group_col='device', test_size=0.30, random_state=42)

# 4) Verificar distribución por failure en cada muestra

def dist(y):
    v = pd.Series(y).value_counts(normalize=True).sort_index()
    return {int(k): float(v.get(k, 0.0)) for k in [0,1]}

print("\nDistribución por failure (proporción):")
print({
    'train': dist(train_df['failure']),
    'test': dist(test_df['failure']),
    'control': dist(control_df['failure']),
    'oos': dist(oos_df['failure'])
})

# Reporte final
print("\nResumen de tamaños:")
print({
    'train_rows': len(train_df),
    'test_rows': len(test_df),
    'control_rows': len(control_df),
    'oos_rows': len(oos_df)
})

# Variables de salida clave
SPLITS = {
    'train': train_df,
    'test': test_df,
    'control': control_df,
    'oos': oos_df
}
print("\nSPLITS = {'train','test','control','oos'} listos.")

Registros periodo train (2015-01..2015-07): 108707
Registros periodo OOT (2015-08..2015-11): 15787
Devices OOT totales: 334
Devices control seleccionados: 83
Registros training original: 108707
Registros training después de excluir control: 91838
Registros control: 3635 | Registros OOS: 12152

Distribución por failure (proporción):
{'train': {0: 0.9989284156750359, 1: 0.0010715843249641767}, 'test': {0: 0.9988776655443322, 1: 0.001122334455667789}, 'control': {0: 0.9988995873452545, 1: 0.0011004126547455295}, 'oos': {0: 0.9997531270572745, 1: 0.0002468729427254773}}

Resumen de tamaños:
{'train_rows': 80255, 'test_rows': 11583, 'control_rows': 3635, 'oos_rows': 12152}

SPLITS = {'train','test','control','oos'} listos.
Registros periodo train (2015-01..2015-07): 108707
Registros periodo OOT (2015-08..2015-11): 15787
Devices OOT totales: 334
Devices control seleccionados: 83
Registros training original: 108707
Registros training después de excluir control: 91825
Registros control: 4005 |

In [25]:
train_df.failure.value_counts(normalize=True)

failure
0    0.998978
1    0.001022
Name: proportion, dtype: float64

In [26]:
test_df.failure.value_counts(normalize=True)

failure
0    0.998586
1    0.001414
Name: proportion, dtype: float64

In [27]:
control_df.failure.value_counts(normalize=True)

failure
0    0.999501
1    0.000499
Name: proportion, dtype: float64

In [28]:
oos_df.failure.value_counts(normalize=True)

failure
0    0.999576
1    0.000424
Name: proportion, dtype: float64

In [ ]:


def summarize_device_overlaps(train_df=None, test_df=None, control_df=None, oos_df=None, splits=None):
    """Resumen de solapamientos de devices entre conjuntos creados."""
    # Armar diccionario de datasets
    if splits is not None:
        datasets = {k: v for k, v in splits.items() if v is not None and len(v) > 0}
    else:
        datasets = {}
        if train_df is not None: datasets['train'] = train_df
        if test_df is not None: datasets['test'] = test_df
        if control_df is not None: datasets['control'] = control_df
        if oos_df is not None: datasets['oos'] = oos_df

    if not datasets:
        print("No hay datasets para analizar.")
        return

    # Conjuntos de devices por dataset
    device_sets = {name: set(df['device'].unique()) for name, df in datasets.items()}

    # Tamaños por dataset
    sizes = {name: len(devs) for name, devs in device_sets.items()}
    print("=== Devices únicos por dataset ===")
    print(sizes)

    # Matriz de intersecciones (conteo)
    names = list(device_sets.keys())
    inter_matrix = pd.DataFrame(index=names, columns=names, dtype=int)
    for a in names:
        for b in names:
            inter_matrix.loc[a, b] = len(device_sets[a] & device_sets[b])

    print("\n=== Intersecciones (conteo de devices en común) ===")
    print(inter_matrix)

    # Porcentaje de solapamiento relativo al conjunto de la fila
    pct_matrix = inter_matrix.copy().astype(float)
    for a in names:
        denom = max(1, sizes[a])
        pct_matrix.loc[a] = (pct_matrix.loc[a] / denom) * 100.0

    print("\n=== Intersecciones (% relativo a la fila) ===")
    print(pct_matrix.round(1))

    # Reglas simples de leakage
    leakage_rules = {}
    if 'train' in device_sets and 'test' in device_sets:
        leakage_rules['train_test'] = len(device_sets['train'] & device_sets['test']) > 0
    if 'train' in device_sets and 'control' in device_sets:
        leakage_rules['train_control'] = len(device_sets['train'] & device_sets['control']) > 0
    if 'test' in device_sets and 'control' in device_sets:
        leakage_rules['test_control'] = len(device_sets['test'] & device_sets['control']) > 0
    if 'control' in device_sets and 'oos' in device_sets:
        leakage_rules['control_oos'] = len(device_sets['control'] & device_sets['oos']) > 0

    if leakage_rules:
        print("\n=== Verificación de leakage (debe ser False) ===")
        print(leakage_rules)

    # Resumen general
    union_all = set().union(*device_sets.values())
    inter_any = 0
    if len(names) >= 2:
        # Conteo de devices que aparecen en 2+ conjuntos
        from collections import Counter
        counts = Counter()
        for devset in device_sets.values():
            for d in devset:
                counts[d] += 1
        inter_any = sum(1 for v in counts.values() if v >= 2)

    print("\n=== Resumen general ===")
    print({
        'num_datasets': len(datasets),
        'devices_totales_unicos': len(union_all),
        'devices_en_2_o_mas_sets': inter_any
    })


#summarize_device_overlaps(splits=SPLITS)
summarize_device_overlaps(train_df=train_df, test_df=test_df, control_df=control_df, oos_df=oos_df)


=== Devices únicos por dataset ===
{'train': 975, 'test': 111, 'control': 83, 'oos': 251}

=== Intersecciones (conteo de devices en común) ===
         train   test  control    oos
train    975.0    0.0      0.0  221.0
test       0.0  111.0      0.0   30.0
control    0.0    0.0     83.0    0.0
oos      221.0   30.0      0.0  251.0

=== Intersecciones (% relativo a la fila) ===
         train   test  control    oos
train    100.0    0.0      0.0   22.7
test       0.0  100.0      0.0   27.0
control    0.0    0.0    100.0    0.0
oos       88.0   12.0      0.0  100.0

=== Verificación de leakage (debe ser False) ===
{'train_test': False, 'train_control': False, 'test_control': False, 'control_oos': False}

=== Resumen general ===
{'num_datasets': 4, 'devices_totales_unicos': 1169, 'devices_en_2_o_mas_sets': 251}
=== Devices únicos por dataset ===
{'train': 975, 'test': 111, 'control': 83, 'oos': 251}

=== Intersecciones (conteo de devices en común) ===
         train   test  control    oos

In [32]:
train_df.to_parquet('C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_train.parquet', engine='pyarrow')
test_df.to_parquet('C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_test.parquet', engine='pyarrow')
control_df.to_parquet('C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_oot.parquet', engine='pyarrow')
oos_df.to_parquet('C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_oos.parquet', engine='pyarrow')
